In [112]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

In [113]:
class GridWorld2D(gym.Env):

    def __init__(self, grid_size=5):
        super().__init__()
        self.grid_size = grid_size
        self.action_space = spaces.Discrete(4) # 0=Up, 1=Down, 2=Left, 3=Right
        self.goal_pos = np.array([grid_size-1, grid_size-1])
        
        self.obstacles = {(1, 1), (2, 2),(1,3)}

    def reset(self, seed=None):
        super().reset(seed=seed)
        self.agent_pos = np.array([0, 0], dtype=np.int32)
        return self.agent_pos, {}

    def step(self, action):
        x, y = self.agent_pos
        nx, ny = x, y

        if action == 0: nx -= 1
        elif action == 1: nx += 1
        elif action == 2: ny -= 1
        elif action == 3: ny += 1

        nx = np.clip(nx, 0, self.grid_size - 1)
        ny = np.clip(ny, 0, self.grid_size - 1)

        if (nx, ny) in self.obstacles:
            nx, ny = x, y
            reward = -10

        self.agent_pos = np.array([nx, ny])
        terminated = False
        

        if np.array_equal(self.agent_pos, self.goal_pos):
            reward = 100
            terminated = True
        else:
            reward = -1
            
        return self.agent_pos, reward, terminated


In [114]:
def run_training():
    env = GridWorld2D(grid_size=5)
    grid_size = 5
    n_states = grid_size * grid_size
    n_actions = 4
    Q = np.zeros((n_states, n_actions))
    
    # Learning Parameters
    episodes = 1000 
    alpha = 0.1
    gamma = 0.9
    epsilon = 0.2

    for _ in range(episodes):
        state, _ = env.reset()
        state_idx = state[0] * grid_size + state[1]
        done = False

        while not done:
            if np.random.rand() < epsilon:
                action = env.action_space.sample()
            else:
                action = np.argmax(Q[state_idx])

            next_state, reward, terminated = env.step(action)
            next_idx = next_state[0] * grid_size + next_state[1]
            
            # Q-Update
            best_next = np.max(Q[next_idx])
            Q[state_idx, action] += alpha * (reward + gamma * best_next - Q[state_idx, action])
            
            state_idx = next_idx
            done = terminated

    
    return Q


In [115]:
def print_simple_visuals(env, Q):
    grid_size = env.grid_size
    
    arrows = ["↑", "↓", "←", "→"]
    
    print("BEST POLICY")

    for r in range(grid_size):
        row_string = ""
        for c in range(grid_size):
            state_idx = r * grid_size + c
            
            if (r, c) == (0, 0):
                char = "S"
            elif (r, c) == (grid_size - 1, grid_size - 1):
                char = "G"
            elif (r, c) in env.obstacles:
                char = "X"
            else:
                best_action = np.argmax(Q[state_idx])
                char = arrows[best_action]
            
            row_string += f" {char} "
        print(row_string)


In [116]:
env = GridWorld2D(grid_size=5)
    
Q_table = run_training()
    
Q_table

array([[ 3.72645925e+01,  4.26126590e+01,  3.73209318e+01,
         3.23124174e+01],
       [ 5.68559392e+00,  8.69718913e+00,  3.73032814e+01,
        -2.45641266e+00],
       [-2.03634420e+00, -1.97289219e+00, -1.95625604e+00,
        -1.74483975e+00],
       [-1.39869848e+00, -1.35153655e+00, -1.46545601e+00,
         3.74937790e-01],
       [-8.34797216e-01,  6.34684149e+00, -1.01766717e+00,
        -9.50056932e-01],
       [ 3.72412422e+01,  4.84585100e+01,  4.25254479e+01,
         4.25514037e+01],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [-2.01853182e+00, -1.95188176e+00, -1.98369410e+00,
        -1.97202898e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [-5.20243301e-01,  2.21060801e+01, -4.81366810e-01,
        -5.68199531e-01],
       [ 4.23550177e+01,  5.49539000e+01,  4.83399315e+01,
         5.44699141e+01],
       [ 2.13363230e+01,  6.21684679e+01,  1.92128976e+01,
      

In [117]:
print_simple_visuals(env, Q_table)

BEST POLICY
 S  ←  →  →  ↓ 
 ↓  X  ↓  X  ↓ 
 ↓  ↓  X  ↓  ↓ 
 →  ↓  ↓  ↓  ↓ 
 →  →  →  →  G 
